# Vertical gradients — where the centred stencil loses information

The vertical counterpart of
[`field_validation_sparkle.ipynb`](field_validation_sparkle.ipynb).
That notebook took apart the HORIZONTAL artifact — interpolating a
finite difference across the axis it was differenced along.  This one
takes apart the vertical one, which is not the same thing and needs
its own argument.

| | |
|---|---|
| Domain | one 720 × 720 × 51 tile (Section 1) |
| Applies to | every field downstream of `∂/∂z`: N², Ri, vertical_shear, Fr, Bu, R_ib, ertel_pv |
| Companions | `docs/Gradients.md`, `prompts/field_validation_depth.md` §4b |

## The claim, and the correction to it

`vertical_helpers._vertical_derivative` is **centred** at interior
levels:

    (f[k+1] − f[k−1]) / (z[k+1] − z[k−1])

On even spacing that is identically the mean of the two one-sided
slopes either side of level k.  So it never reads level k, and the
first thing anyone notices is that this looks exactly like the
horizontal sparkle with the interpolation baked into the stencil.

**That framing is half wrong, and the half that is wrong matters.**
Averaging two one-sided slopes does two different things depending on
their signs:

- **Same sign, different magnitude** — a monotone *kink*, e.g. the base
  of the mixed layer where the gradient goes from ~0 above to large
  below.  The centred form returns roughly their mean.  That is a
  correct second-order estimate at a place where the derivative is
  genuinely not well defined.  **Not an error.**
- **Opposite signs** — level k is a vertical *extremum*: an inversion,
  a spike, a one-level step.  The two slopes partly annihilate, the
  centred form collapses toward zero while both one-sided slopes stay
  large.  **This is the artifact**, and it is the real vertical
  analogue of the sparkle.

A metric that does not separate these two overstates the problem
badly, because the mixed-layer base is a kink almost everywhere and
will light up the whole map.  Sections 3 and 4 measure them apart.


## Section 1 — Setup: one tile, full water column

In [ ]:
# ---- knobs -------------------------------------------------------------
REGION       = "gulf_stream"          # any region with a 'zoom' anchor
DATE         = "2012-11-09 12:00:00"
LEVELS       = ("sfc", "z25m", "mld", "mld_mean")
ZOOM_HALF_KM = 100.0
# ------------------------------------------------------------------------

import dask
import numpy as np
import xarray as xr

import dbof.preprocessing.calculate_fields as CF
import dbof.preprocessing.calculate_fields_at_depth as CFAD
from dbof.plotting import depth_figures as dfig
from dbof.plotting.field_cmaps import load_field_cmaps
import dbof.utils.native_gradient as NG
from dbof.preprocessing import vertical_helpers as VH
from dbof.tiles import tile_utils
from dbof.tiles.tile_mapping import rect_ij_to_tile

# tile_utils sets the Agg backend on import; restore inline afterwards.
%matplotlib inline
import matplotlib.pyplot as plt

CMAP_CFG, DIVERGING = load_field_cmaps()

S3 = tile_utils._resolve_s3_source(None)
ANCHOR_LON, ANCHOR_LAT = dfig.region_anchor(REGION)
tile = rect_ij_to_tile(
    *tile_utils.latlon_to_rect_ij(ANCHOR_LON, ANCHOR_LAT, S3))
print(f"tile   : idx {tile.tile_idx}, face {tile.face_idx}")

ds_grid = tile_utils._load_grid_for_tile(S3, tile)
ds_raw = tile_utils._load_tracers_for_tile(
    S3, DATE, tile, ["Theta", "Salt", "U", "V", "W"])
ds_merge, xgrid = tile_utils._build_tile_context(ds_raw, ds_grid)

XC, YC = dfig.tile_coords(ds_grid)
LAND = dfig.tile_land_mask(ds_grid)

# Depth coordinate, positive downward, and the layer thicknesses.
Z = np.asarray(VH._get_depth_coord(ds_merge).values, dtype=float)
print(f"levels : {len(Z)}, {Z[0]:.1f} m to {Z[-1]:.1f} m")

# Two lazy fields every section below needs.
rho = CF.potential_density(ds_merge)
mld = CFAD.mixed_layer_depth(ds_merge)

## Section 2 — The mechanism, on one synthetic column

Before touching model data: three profiles where the right answer is
known by construction.  If the diagnostic cannot separate these, it
cannot be trusted on the tile.


In [ ]:
# Three profiles on the real model levels.
rho_lin = 1025.0 + 0.004 * Z                       # smooth, no kink
rho_kink = 1025.0 + 0.02 * np.clip(Z - 120.0, 0, None) / 10.0
rho_spike = rho_lin.copy()
rho_spike[20] += 0.35                              # one-level inversion


def stencil_1d(rho):
    """centred, the two one-sided slopes, asym and sign flip."""
    n = len(rho)
    c = np.full(n, np.nan)
    c[1:-1] = (rho[2:] - rho[:-2]) / (Z[2:] - Z[:-2])
    above = np.full(n, np.nan)
    below = np.full(n, np.nan)
    above[1:] = (rho[1:] - rho[:-1]) / (Z[1:] - Z[:-1])
    below[:-1] = (rho[1:] - rho[:-1]) / (Z[1:] - Z[:-1])
    bigger = np.where(np.abs(above) >= np.abs(below), above, below)
    with np.errstate(invalid="ignore", divide="ignore"):
        asym = 1.0 - np.abs(c) / np.abs(bigger)
    flip = (above * below) < 0
    return c, bigger, asym, flip


print(f"{'profile':<12}{'max asym':>10}{'sign flips':>12}  verdict")
print("-" * 62)
for nm, r in (("linear", rho_lin), ("kink", rho_kink),
              ("spike", rho_spike)):
    c, b, a, fl = stencil_1d(r)
    print(f"{nm:<12}{np.nanmax(a):>10.2f}{int(np.nansum(fl)):>12}  "
          + ("clean" if np.nanmax(a) < 0.05 else
             ("CANCELLATION" if fl.any() else "curvature only")))

The kink should show a large asymmetry and **zero** sign flips — the
stencil is averaging across a curvature, which is what a centred
difference is supposed to do.  The spike should show sign flips.  That
is the whole distinction, and everything below rests on it.


## Section 3 — The same diagnostic on the tile, on potential density

`dfig.vertical_stencil_ab` computes all four fields lazily on the full
3D volume, then they are reduced to the four depth levels the pipeline
stores.

**How `dz_signflip` reduces matters.**  At `sfc`, `z25m` and `mld` the
depth strategy picks a single level, so the flag stays 0 or 1 and its
mean is the fraction of cells affected.  At `mld_mean` it is
thickness-weighted over the whole layer, so it arrives as a fraction in
[0, 1] and is essentially never exactly 1.  Thresholding it reports
0.0% for a level that is genuinely affected — so it is **averaged, not
thresholded**, and the number means "fraction of the mixed layer
affected" rather than "fraction of cells flagged".


In [ ]:
AB = dfig.vertical_stencil_ab(rho, ds_merge)
ab = dfig.pack_tile_levels(
    dfig.compute_levels(AB, ds_merge, mld=mld, levels=LEVELS),
    XC, YC, edge_margin=0, land_mask=LAND, levels=LEVELS,
    verbose=False)

dfig.depth_map_grid(
    ["dz_centred", "dz_onesided", "dz_asym", "dz_signflip"], ab,
    CMAP_CFG, region=REGION, levels=LEVELS,
    diverging_cmaps=DIVERGING, zoom_half_km=ZOOM_HALF_KM,
    suptitle=("Figure 1 — centred vs one-sided ∂ρ/∂z, with curvature "
              "(asym) and cancellation (signflip) separated"))
plt.show()

print(f"{'level':<10}{'median asym':>13}{'asym>0.25':>11}"
      f"{'% flipped':>11}")
print("-" * 45)
for lev in LEVELS:
    a, sf = ab["dz_asym"][lev][2], ab["dz_signflip"][lev][2]
    if not np.isfinite(a).any():
        continue
    print(f"{lev:<10}{np.nanmedian(a):>13.3f}"
          f"{100 * np.nanmean(a > 0.25):>10.1f}%"
          f"{100 * np.nanmean(sf):>10.1f}%")
print("")
print("asym = curvature (benign at a kink).")
print("% flipped = true cancellation; at mld_mean it is the "
      "thickness-weighted")
print("            fraction of the layer, not a per-cell flag.")

## Section 4 — Does the answer depend on which field you differentiate?

Section 3 answers for potential density.  But the pipeline takes `∂/∂z`
of several different things, and a field that is **already a
derivative** is rougher in the vertical than the tracer it came from —
so its vertical derivative should cancel more often.  That is the
hypothesis; this section tests it across six representative fields.


In [ ]:
# Six fields spanning the kinds of thing the pipeline computes, so the
# diagnostics below are not answered on potential density alone.
#
#   Theta              raw tracer, vertically smooth
#   b                  buoyancy: a scaled tracer, so also smooth
#   gradb2             HORIZONTAL gradient (squared before interp)
#   relative_vorticity HORIZONTAL Jacobian (ECCO recipe)
#   N2                 VERTICAL gradient -- already one d/dz deep
#   ertel_pv           product of a VERTICAL and a HORIZONTAL gradient
#
# The last two matter most: a field that is already a derivative is
# rougher in the vertical than the tracer it came from, so ITS vertical
# derivative cancels more often.
jac = CF.compute_velocity_jacobian(ds_merge, xgrid)

ZOO = {
    "Theta": ds_merge["Theta"],
    "b": CF.buoyancy_of_field(ds_merge),
    "gradb2": CF.grad_b2(ds_merge, xgrid),
    "relative_vorticity": CF.relative_vorticity(
        ds_merge, xgrid, jacobian=jac),
    "N2": CFAD.buoyancy_frequency_squared(ds_merge),
    "ertel_pv": CFAD.ertel_pv_terms(ds_merge, xgrid)["ertel_pv"],
}
ZOO_KIND = {
    "Theta": "raw tracer",
    "b": "scaled tracer",
    "gradb2": "horizontal gradient (squared first)",
    "relative_vorticity": "horizontal Jacobian",
    "N2": "vertical gradient",
    "ertel_pv": "vertical x horizontal product",
}
print("field zoo ready:", list(ZOO))

In [ ]:
# Sign-flip fraction and median asymmetry per level, for every field.
rows = []
for nm, fld in ZOO.items():
    ab_f = dfig.pack_tile_levels(
        dfig.compute_levels(
            dfig.vertical_stencil_ab(fld, ds_merge),
            ds_merge, mld=mld, levels=LEVELS),
        XC, YC, edge_margin=3, land_mask=LAND, levels=LEVELS,
        verbose=False)
    rows.append((nm, ab_f))
    print(f"  done: {nm}")

print("")
print(f"{'field':<20}{'kind':<36}" + "".join(f"{l:>11}" for l in LEVELS))
print("-" * (56 + 11 * len(LEVELS)))
for nm, ab_f in rows:
    cells_pct = "".join(
        f"{100 * np.nanmean(ab_f['dz_signflip'][l][2]):>10.1f}%"
        for l in LEVELS)
    print(f"{nm:<20}{ZOO_KIND[nm]:<36}{cells_pct}")
print("")
print("% of the tile where ∂/∂z of that field straddles a vertical "
      "extremum.")

In [ ]:
# The same thing as a depth profile, all six fields on one axis.
fig, ax = plt.subplots(figsize=(7.5, 7))
for (nm, _), c in zip(rows, dfig.LOCATION_COLORS + ("0.35",)):
    sf3 = dfig.vertical_stencil_ab(ZOO[nm], ds_merge)["dz_signflip"]
    frac = dask.compute(
        sf3.mean(dim=[d for d in sf3.dims if d != "k"]))[0]
    ax.plot(100 * np.asarray(frac.values), Z, color=c, linewidth=2,
            label=f"{nm} ({ZOO_KIND[nm]})")
ax.set_ylim(Z.max(), 0)
ax.set_xlabel("% of tile with a sign flip in ∂/∂z", fontsize=9)
ax.set_ylabel("depth (m)", fontsize=9)
ax.grid(alpha=0.25, linewidth=0.6)
ax.legend(fontsize=8, loc="lower right")
fig.suptitle("Figure 2 — cancellation by field type and depth "
             "(surface at top)", fontsize=12)
fig.tight_layout()
plt.show()

**What to conclude.**  If `Theta` and `b` sit low while `N2`,
`relative_vorticity` and `ertel_pv` sit much higher, then the vertical
stencil is not really a property of the discretisation alone — it is a
property of *how many derivatives deep* a field already is.  That has a
direct consequence: taking `∂/∂z` of an already-differentiated field is
the expensive operation, and `ertel_pv`, which does exactly that and
then multiplies across directions, should be the worst.

It also means the fix, if there is one, belongs upstream: smoothing or
re-staggering `N2` would help everything below it, while patching
`ertel_pv` alone would not.


## Section 5 — Order of operations: reduce-then-combine vs combine-then-reduce

Everything above is about one derivative.  This section is about what
happens when two of them meet, which is where the depth pipeline has a
real choice to make.

`ertel_pv`'s tilting term contains products like `v_z · b_x` — a
**vertical** gradient times a **horizontal** one.  The pipeline computes
the product in 3D and reduces afterwards.  The alternative is to reduce
each factor to the level first and multiply the 2D results:

- **A (production)**: `reduce(v_z · b_x)`
- **B**: `reduce(v_z) · reduce(b_x)`

For `sfc`, `z25m` and `mld` these are **identical** — the strategy picks
one level, and picking a level commutes with multiplication.

For `mld_mean` they are **not**, because a thickness-weighted mean does
not commute with a product: `mean(a·b) ≠ mean(a)·mean(b)`.  The
difference is the covariance of `a` and `b` over the mixed layer, which
is a real physical quantity, not an error.  A is the mixed-layer
average of the flux; B is the flux of the mixed-layer averages.  They
answer different questions and the pipeline should be explicit about
which one it means.


In [ ]:
b = CF.buoyancy_of_field(ds_merge)
b_x, b_y = NG.calculate_native_gradient_tracer(b, ds_merge, grid=xgrid)
u_z, v_z = CFAD.vertical_shear_components(ds_merge, xgrid)

term = v_z * b_x                      # one tilting-term factor pair

# A: combine in 3D, then reduce.  B: reduce each, then combine.
A = dfig.compute_levels({"A": term}, ds_merge, mld=mld, levels=LEVELS)
P = dfig.compute_levels({"vz": v_z, "bx": b_x}, ds_merge, mld=mld,
                        levels=LEVELS)
B = {f"B_{l}": P[f"vz_{l}"] * P[f"bx_{l}"] for l in LEVELS}

order = dfig.pack_tile_levels(
    {**{f"combine_then_reduce_{l}": A[f"A_{l}"] for l in LEVELS},
     **{f"reduce_then_combine_{l}": B[f"B_{l}"] for l in LEVELS}},
    XC, YC, edge_margin=3, land_mask=LAND, levels=LEVELS,
    verbose=False)

print(f"{'level':<12}{'max |A-B| / rms(A)':>22}  verdict")
print("-" * 56)
for lev in LEVELS:
    a = order["combine_then_reduce"][lev][2]
    bb = order["reduce_then_combine"][lev][2]
    rel = np.nanmax(np.abs(a - bb)) / max(
        float(np.sqrt(np.nanmean(a ** 2))), 1e-30)
    same = rel < 1e-4
    print(f"{lev:<12}{rel:>22.3e}  "
          + ("identical (as expected)" if same
             else "DIFFERENT -- covariance term"))

In [ ]:
dfig.depth_map_grid(
    ["combine_then_reduce", "reduce_then_combine"], order, CMAP_CFG,
    region=REGION, levels=LEVELS, diverging_cmaps=DIVERGING,
    zoom_half_km=ZOOM_HALF_KM,
    suptitle=("Figure 3 — v_z · b_x: combining in 3D then reducing "
              "(production) vs reducing then combining"))
plt.show()

## Section 6 — Decision

Fill in from the numbers above.  Three separable questions:

1. **Is the sign-flip rate material?**  Section 3 gives it for density;
   Section 4 gives it per field type.  A few percent means isolated bad
   pixels in `Ri`, `Fr`, `Bu` and `ertel_pv` — worth annotating, not
   worth re-plumbing.  Ten percent or more at the MLD, on the fields
   that feed the sampled products, is a different conversation.
2. **Does it concentrate where we sample?**  Figure 2 answers this.  If
   the peak sits near typical MLD depths, the `_mld` row is sampling
   the worst part of the column and that is worth stating in every
   subset notebook.
3. **Which order do we mean for `mld_mean`?**  Section 5 shows the two
   are genuinely different for products.  Production computes
   `mean(a·b)`, the mixed-layer average of the quantity — which is
   almost certainly the intended meaning, but it has never been written
   down, and `docs/Fields.md` should say so.

### If a stencil fix is ever warranted

The vertical grid is staggered: `k_l` levels sit between tracer levels.
A one-sided difference lands naturally on `k_l`, so the vertical
analogue of the horizontal square-before-interp fix does exist —
compute `∂ρ/∂z` on `k_l` and interpolate to `k` only after whatever
squaring the consumer needs.  That changes `N2` and everything below
it, so it is not a change to make on the strength of a map.  Make it on
the strength of Section 4.

---

### Cross-references

- **The horizontal artifact** — `docs/Gradients.md`,
  `field_validation_sparkle.ipynb`.
- **The MLD staircase**, a different mechanism with similar symptoms —
  `mixed_layer_depth.ipynb`.
- **The fields affected** — `depth_fields/vertical_shear.ipynb`,
  `mixing_parameters.ipynb`, `ertel_pv.ipynb`.
